# Step 01 - Segmentation and Feature Extraction

This notebook runs the Step 1 pipeline: load sequence IDs, segment traces into steps, extract top-k SAE features per step, and save/load a cached step-feature table.

1. Clone/pull the project from the public GitHub repo and set up imports
2. Download/connect databases
3. Load cached sequence IDs
4. Build or load the Step 1 checkpoint table
5. Run quick validation/inspection cells

In [ ]:
# Cell 1 - GitHub setup + imports
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/shreetishresthanp/latent_verbal_gap.git"
WORKDIR = Path("/content")
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
REPO = WORKDIR / REPO_NAME

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated existing repo: {REPO}")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    print(f"Cloned repo: {REPO}")

REQ_FILE = REPO / "requirements.txt"
SRC_DIR = REPO / "src"

if REQ_FILE.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)],
        check=True,
    )
else:
    print(f"Skipping install: missing {REQ_FILE}")

src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

Updated existing repo: /content/latent_verbal_gap
Skipping install: missing /content/latent_verbal_gap/requirements.txt


In [ ]:
from config import CACHE_DIR
from segmentation import build_step_feature_table
from utils import connect_dbs, download_databases, load_seq_ids, set_seeds

set_seeds()
print(f"Using src at: {SRC_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")

In [28]:
# Reference only (disabled): legacy in-notebook implementation.
# The active implementation is imported from src/segmentation.py via
# `from segmentation import build_step_feature_table`.
# Keep this block commented for historical comparison only.

In [ ]:
# Downloading open sourced DBs (skip if already on VM)
download_databases()
conn = connect_dbs()

  → saved to /root/math-autointerp.db
  → saved to /root/math-0-1.ddb


In [ ]:
# Loading seq_ids
print(CACHE_DIR)
seq_ids = load_seq_ids(conn, CACHE_DIR / "seq_ids_with_activations.json")

/content/drive/MyDrive/latent_verbal_gap/checkpoints
Loaded 93 seq_ids from cache


In [ ]:
# Building or Loading step, feature pairs table
import pandas as pd
_ckpt = CACHE_DIR / "step_feat_df_step1_maxpool_k3_lnorm.parquet"
FORCE_REBUILD = False

if _ckpt.exists() and not FORCE_REBUILD:
    step_feat_df = pd.read_parquet(_ckpt)
    print(f"Loaded from cache: {step_feat_df.shape}")
else:
    step_feat_df = build_step_feature_table(conn, seq_ids, k=3)
    step_feat_df.to_parquet(_ckpt, index=False)
    print(f"Built and saved: {step_feat_df.shape}")

step_feat_df.head()

Loaded from cache: (64554, 12)


,sequence_id,step_idx,start_token,end_token,step_text,step_type,rel_step_pos,feature_rank,feature_id,feature_label,strength,strength_w
0,1000,0,64,76,"Alright, let's tackle this physics problem.",setup_planning,0.000000,1,26742,Mathematical problem-solving with formal notation,3.125000,0.373134
1,1000,0,64,76,"Alright, let's tackle this physics problem.",setup_planning,0.000000,2,22966,Beginning of mathematical problem-solving thou...,2.718750,0.324627
2,1000,0,64,76,"Alright, let's tackle this physics problem.",setup_planning,0.000000,3,10219,Mathematical word problems requiring calculati...,2.531250,0.302239
3,1000,1,76,98,The question is asking what magnitude force Jo...,other,0.006993,1,15072,Mathematical reasoning and step-by-step proble...,2.906250,0.358382
4,1000,1,76,98,The question is asking what magnitude force Jo...,other,0.006993,2,26742,Mathematical problem-solving with formal notation,2.640625,0.325626


In [ ]:
# Inspecting values for sanity check
n_steps = step_feat_df[["sequence_id","step_idx"]].drop_duplicates().shape[0]
print(f"Sequences: {step_feat_df['sequence_id'].nunique()}")
print(f"Unique steps: {n_steps}")
print(f"\nStep type distribution:")
print(step_feat_df.drop_duplicates(["sequence_id","step_idx"])["step_type"]
      .value_counts().to_string())

Sequences: 93
Unique steps: 21518

Step type distribution:
step_type
other              10112
calculation         6180
setup_planning      1829
self_correction     1805
uncertainty         1592


In [10]:
print(step_feat_df.shape)
print(step_feat_df["feature_rank"].value_counts().sort_index())
print(f"\nUnique sequences: {step_feat_df['sequence_id'].nunique()}")
print(f"Unique steps: {step_feat_df[['sequence_id','step_idx']].drop_duplicates().shape[0]}")
print(step_feat_df.head())

(64554, 12)
feature_rank
1    21518
2    21518
3    21518
Name: count, dtype: int64

Unique sequences: 93
Unique steps: 21518
   sequence_id  step_idx  start_token  end_token  \
0         1000         0           64         76   
1         1000         0           64         76   
2         1000         0           64         76   
3         1000         1           76         98   
4         1000         1           76         98   

                                           step_text       step_type  \
0        Alright, let's tackle this physics problem.  setup_planning   
1        Alright, let's tackle this physics problem.  setup_planning   
2        Alright, let's tackle this physics problem.  setup_planning   
3  The question is asking what magnitude force Jo...           other   
4  The question is asking what magnitude force Jo...           other   

   rel_step_pos  feature_rank  feature_id  \
0      0.000000             1       26742   
1      0.000000             2       22

In [11]:
from scipy.stats import spearmanr

step_feat_df["step_length"] = step_feat_df["end_token"] - step_feat_df["start_token"]
r, p = spearmanr(
    step_feat_df.drop_duplicates(["sequence_id","step_idx"])["step_length"],
    step_feat_df.groupby(["sequence_id","step_idx"])["strength_w"].mean().values
)
print(f"Step length vs strength_w: r={r:.3f}, p={p:.4f}")
# Should be near 0 now

Step length vs strength_w: r=nan, p=nan


/tmp/ipykernel_561/2321610320.py:4: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = spearmanr(


In [12]:
# Check within-step variation in strength_w
print("strength_w distribution:")
print(step_feat_df["strength_w"].describe().round(4))

print("\nSample of one step's weights:")
sample_step = step_feat_df.groupby(["sequence_id","step_idx"]).first().index[0]
print(step_feat_df[
    (step_feat_df["sequence_id"] == sample_step[0]) &
    (step_feat_df["step_idx"]    == sample_step[1])
][["feature_rank","strength","step_length","strength_w"]])

# Check if strength_w sums to 1 per step (should always be true)
sums = step_feat_df.groupby(["sequence_id","step_idx"])["strength_w"].sum()
print(f"\nstrength_w sums to 1 per step: {sums.between(0.99, 1.01).all()}")

strength_w distribution:
count    64554.0000
mean         0.3333
std          0.0992
min          0.1335
25%          0.2530
50%          0.3032
75%          0.4163
max          0.6376
Name: strength_w, dtype: float64

Sample of one step's weights:
   feature_rank  strength  step_length  strength_w
0             1   3.12500           12    0.373134
1             2   2.71875           12    0.324627
2             3   2.53125           12    0.302239

strength_w sums to 1 per step: True


In [12]:
r, p = spearmanr(
    step_feat_df.drop_duplicates(["sequence_id","step_idx"])["step_length"],
    step_feat_df.groupby(["sequence_id","step_idx"])["strength"].mean().values
)
print(f"Step length vs raw strength: r={r:.3f}, p={p:.4f}")

Step length vs raw strength: r=0.521, p=0.0000
